# STIR-Net V1 — 17 notebook-patched event-aware competitive temporal routing

Notebook 16 showed that the five-frame evidence is present, but both the all-pairs GNN and the split-query fine-node reader are very diffuse. This notebook tests a **narrow downstream intervention without editing `learned/stirnet/`**.

The experiment starts from Notebook-15 `checkpoint_temporal_dense.pt` at step 50 and patches only the temporal reader at runtime:

- **event-aware node relevance**: interior starts/newborns, interior ends/broken tracks, and divisions receive a higher learnable correction-relevance prior; boundary events are down-weighted; continuous tracks stay available;
- **source-group competition**: `primary + split0 + split1 + ...` from the same current component jointly compete for fine temporal nodes;
- **tracklet memory stays unchanged** as broad temporal context;
- **Detection-GNN architecture stays unchanged** for this experiment.

After 35 training steps (query bootstrap → native bootstrap → joint), the same trained weights are evaluated under:

- `full` = event relevance + competition
- `no_event` = competition only
- `no_competition` = event relevance only
- `baseline` = the original independent reader
- `shuffled_event` = event relevance attached to the wrong temporal nodes

A successful result must show not only more diverse attention, but also better source-9 center/mask specialization.

In [ ]:
from pathlib import Path
from dataclasses import replace
from types import MethodType
import gc, json, math, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import _reduced_config, _repo_root, build_real_batch
from learned.stirnet.debugging.probes.matching import run_matching_probe
from learned.stirnet.model.query_builder import QUERY_PRIMARY, QUERY_SPLIT, QUERY_TEMPORAL, QUERY_DISCOVERY
from learned.stirnet.model.heads import dot_mask_logits
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import Trainer, model_forward_from_batch, move_to_device

SEED=40266
SOURCE_ID=9
AMP_DTYPE=torch.float16
LOG_EVERY=5
ROUTING_MODES={'full','no_event','no_competition','baseline','shuffled_event'}

REPO_ROOT=_repo_root(Path.cwd())
DATA_DIR=REPO_ROOT/'data'/'learned'/'stirnet'/'first_overfit'/'BlastoSPIM1_F22_030_034'
NB15_RUN=REPO_ROOT/'runs'/'stirnet'/'first_overfit'/'15_hierarchical_temporal_memory'
STEP50_CHECKPOINT=NB15_RUN/'checkpoint_temporal_dense.pt'
RUN_DIR=REPO_ROOT/'runs'/'stirnet'/'first_overfit'/'17_event_competitive_routing'
RUN_DIR.mkdir(parents=True,exist_ok=True)

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError('Notebook 17 requires CUDA.')
if not STEP50_CHECKPOINT.exists(): raise FileNotFoundError(f'Missing {STEP50_CHECKPOINT}. Run Notebook 15 first.')
device=torch.device('cuda')
print('Repo:',REPO_ROOT);print('GPU:',torch.cuda.get_device_name(0));print('Start:',STEP50_CHECKPOINT)

## 1. Load the full real scene and construct event features

We use only existing graph features. No GT event labels are created.

Current node columns used here:

- `0`: normalized time
- `23,24`: track length before/after
- `27`: current-frame indicator
- `28`: interior start / newborn-like track start
- `29`: interior end / broken-track end
- `30`: division involvement
- `31`: boundary-related

In [ ]:
batch,sample=build_real_batch(DATA_DIR)
target=batch['targets'][0]
cfg=_reduced_config()
cfg.curriculum.enabled=True
cfg.curriculum.spatial_dense_steps=30
cfg.curriculum.temporal_dense_steps=20
cfg.curriculum.query_bootstrap_steps=20
cfg.curriculum.native_bootstrap_steps=10
assert sample['current_count']==36 and sample['target_count']==33 and sample['temporal_tracklets']==52
assert sample['split_companions_by_source'].get(SOURCE_ID)==8
N=int(batch['graph_x'].shape[0]); E=int(batch['graph_edge_index'].shape[1]); assert E==N*(N-1)

gx=batch['graph_x'].detach().float().cpu()
window=2*int(cfg.temporal.temporal_radius)+1
EVENT_FEATURES=torch.stack([gx[:,0],gx[:,23]/window,gx[:,24]/window,gx[:,27],gx[:,28],gx[:,29],gx[:,30],gx[:,31]],dim=-1).float()
START=gx[:,28]>0.5; END=gx[:,29]>0.5; DIV=gx[:,30]>0.5; BOUNDARY=gx[:,31]>0.5
CORRECTION_EVENT=START|END|DIV
CONTINUOUS=(~CORRECTION_EVENT)&(~BOUNDARY)
BASE_EVENT_PRIOR=(-1.0+2.0*START.float()+2.0*END.float()+1.0*DIV.float()-0.75*BOUNDARY.float()).clamp(-3,3)

print('Scene:',sample)
print('nodes/edges/tracklets:',N,E,len(batch['temporal_ref_um']))
display(pd.DataFrame({
    'category':['interior_start','interior_end','division','correction_event_union','boundary','continuous'],
    'count':[int(START.sum()),int(END.sum()),int(DIV.sum()),int(CORRECTION_EVENT.sum()),int(BOUNDARY.sum()),int(CONTINUOUS.sum())],
    'fraction':[float(START.float().mean()),float(END.float().mean()),float(DIV.float().mean()),float(CORRECTION_EVENT.float().mean()),float(BOUNDARY.float().mean()),float(CONTINUOUS.float().mean())],
}))

In [ ]:
def prepare_device_batch(cpu_batch):
    out={}
    for k,v in cpu_batch.items():
        if k=='targets': out[k]=v
        elif k=='spatial_inputs': out[k]=v.to(device=device,dtype=AMP_DTYPE,non_blocking=True)
        elif k=='instance_labels': out[k]=v.to(device=device,dtype=torch.int32,non_blocking=True)
        else: out[k]=move_to_device(v,device)
    return out
b=prepare_device_batch(batch)
model=StirNet(cfg).to(device).eval()
ckpt=load_checkpoint(STEP50_CHECKPOINT,model,optimizer=None,scheduler=None,scaler=None,map_location='cpu',strict=True,migrate_history=True)
print('Loaded step:',ckpt.get('step'))

## 2. Save the unpatched step-50 signature

After installing the notebook patch, `routing_mode='baseline'` must reproduce this output closely.

In [ ]:
@torch.no_grad()
def signature(out):
    return {
        'exist':out.exist_logits.detach().float().cpu(),
        'centers':out.centers_cellscale.detach().float().cpu(),
        'qemb':out.query_embeddings.detach().float().cpu(),
        'coarse_mean':float(out.coarse_mask_logits.detach().float().mean().cpu()),
        'coarse_std':float(out.coarse_mask_logits.detach().float().std().cpu()),
    }
with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
    unpatched=model_forward_from_batch(model,b)
UNPATCHED=signature(unpatched)
del unpatched;gc.collect();torch.cuda.empty_cache()

## 3. Notebook-only event-aware competitive reader

The existing temporal fusion is preserved as `base`. The patch reuses all trained Q/K/V projections, relation bias, gates, tracklet attention and FFN.

For seeded queries of one source, fine-node weights are:

1. ordinary per-query node selection;
2. competition across sibling hypotheses for each node;
3. normalization back over memory.

Event relevance biases the ordinary node selection toward broken/newborn/division evidence while continuous tracks remain available.

In [ ]:
class EventRelevance(nn.Module):
    def __init__(self,features,prior,hidden=32):
        super().__init__();self.register_buffer('features',features.float().clone());self.register_buffer('prior',prior.float().clone())
        self.residual=nn.Sequential(nn.Linear(features.shape[-1],hidden),nn.SiLU(),nn.Linear(hidden,1))
        nn.init.zeros_(self.residual[-1].weight);nn.init.zeros_(self.residual[-1].bias)
    def forward(self):
        r=self.residual(self.features.float()).squeeze(-1);logit=(self.prior+r).clamp(-5,5);return logit,torch.sigmoid(logit),r

class EventCompetitiveFusion(nn.Module):
    def __init__(self,base,cfg,event_features,prior):
        super().__init__();self.base=base;self.cfg=cfg;self.event=EventRelevance(event_features,prior,int(cfg.relation_bias_hidden))
        self.event_strength_raw=nn.Parameter(torch.full((int(cfg.memory_heads),),math.log(math.expm1(1.0)),dtype=torch.float32))
        self.log_temperature=nn.Parameter(torch.tensor(math.log(0.5),dtype=torch.float32));self.mode='full'
    @property
    def event_strength(self): return F.softplus(self.event_strength_raw).clamp(0,4)
    @property
    def temperature(self): return self.log_temperature.exp().clamp(0.15,2.0)
    def set_mode(self,mode):
        if mode not in ROUTING_MODES: raise ValueError(mode)
        self.mode=mode
    def _node_read(self,norm,refs,qbatch,temporal,dref,node_tokens,source_ids,qtypes,return_debug,full_attention):
        attn=self.base.node_attention;mem=temporal.node_memory;Q=len(norm);K=len(node_tokens);out=torch.zeros_like(norm)
        entropy=norm.new_zeros(Q,dtype=torch.float32);maxw=norm.new_zeros(Q,dtype=torch.float32);gsize=torch.ones(Q,device=norm.device,dtype=torch.long)
        topk=min(max(int(attn.debug_topk),0),K);topi=torch.full((Q,topk),-1,device=norm.device,dtype=torch.long);topw=norm.new_zeros((Q,topk),dtype=torch.float32)
        full=norm.new_zeros((Q,K),dtype=torch.float32) if return_debug and full_attention else None
        elogit,eprob,eresid=self.event();
        if self.mode=='shuffled_event': elogit=elogit.roll(1);eprob=eprob.roll(1);eresid=eresid.roll(1)
        use_event=self.mode in {'full','no_competition','shuffled_event'};use_comp=self.mode in {'full','no_event','shuffled_event'}
        for bi in torch.unique(qbatch).tolist():
            qi=torch.nonzero(qbatch==bi,as_tuple=False).flatten();mi=torch.nonzero(mem.batch_index==bi,as_tuple=False).flatten()
            if qi.numel()==0 or mi.numel()==0: continue
            qh=attn._split(attn.q(norm[qi])).permute(1,0,2);kh=attn._split(attn.k(node_tokens[mi])).permute(1,0,2);vh=attn._split(attn.v(node_tokens[mi])).permute(1,0,2)
            content=torch.einsum('hqd,hkd->hqk',qh,kh).float()/math.sqrt(attn.head_dim)
            relation=attn.relation_bias(refs[qi],mem.observed_ref_um[mi],mem.projected_ref_um[mi],mem.time_offset[mi],mem.history_valid[mi],dref[int(bi)]).permute(2,0,1).float()
            logits=content+relation
            selection=logits+(self.event_strength[:,None,None]*elogit[mi][None,None,:] if use_event else 0)
            weights=torch.softmax(selection,dim=-1)
            if use_comp and source_ids is not None and qtypes is not None:
                ls=source_ids[qi];lt=qtypes[qi];seeded=(ls>=0)&((lt==QUERY_PRIMARY)|(lt==QUERY_SPLIT))
                for sid in torch.unique(ls[seeded]).tolist():
                    group=torch.nonzero(seeded&(ls==sid),as_tuple=False).flatten()
                    if group.numel()<=1: continue
                    comp=torch.softmax(logits[:,group,:]/self.temperature,dim=1)
                    w=weights[:,group,:]*comp
                    w=w/w.sum(dim=-1,keepdim=True).clamp_min(1e-12)
                    # Autograd safety: Softmax output is saved for backward.
                    # Replace selected query rows out-of-place rather than
                    # mutating weights[:, group, :] in-place.
                    weights=weights.index_copy(1,group,w)
                    gsize[qi[group]]=int(group.numel())
            msg=torch.einsum('hqk,hkd->hqd',weights,vh.float()).permute(1,0,2).reshape(len(qi),attn.d_model);out[qi]=attn.out(msg.to(norm.dtype)).to(out.dtype)
            if return_debug:
                mw=weights.mean(dim=0);entropy[qi]=-(mw*mw.clamp_min(1e-12).log()).sum(dim=-1);maxw[qi]=mw.max(dim=-1).values
                lk=min(topk,len(mi))
                if lk:
                    lw,li=torch.topk(mw,lk,dim=-1);topi[qi,:lk]=mi[li];topw[qi,:lk]=lw
                if full is not None: full[qi[:,None],mi[None,:]]=mw
        debug=None
        if return_debug:
            debug={'entropy':entropy.detach(),'max_weight':maxw.detach(),'top_indices':topi.detach(),'top_weights':topw.detach(),'query_batch_index':qbatch.detach(),'memory_count':torch.tensor(K,device=norm.device),'competition_group_size':gsize.detach(),'event_logit':elogit.detach(),'event_probability':eprob.detach(),'event_residual_logit':eresid.detach(),'event_strength':self.event_strength.detach(),'competition_temperature':self.temperature.detach(),'routing_mode':self.mode}
            if full is not None: debug['full_weights']=full.detach()
        return out,debug
    def forward(self,query_tokens,query_ref_um,query_batch_index,temporal,dref_um,*,memory_ablation='full',return_debug=False,full_attention=False,query_source_ids=None,query_types=None):
        norm=self.base.norm(query_tokens);out=query_tokens;diag={};used=False;mem=temporal.node_memory
        if memory_ablation!='tracklet_only' and mem is not None and not mem.is_empty:
            nt=mem.tokens
            if memory_ablation=='zero_node': nt=torch.zeros_like(nt)
            elif memory_ablation=='shuffle_node': nt=self.base._shuffle_node_tokens(mem)
            nmsg,ndbg=self._node_read(norm,query_ref_um,query_batch_index,temporal,dref_um,nt,query_source_ids,query_types,return_debug,full_attention)
            gate=torch.sigmoid(self.base.node_gate(torch.cat([norm,nmsg],dim=-1)));out=out+gate*nmsg;used=True
            if ndbg is not None: ndbg['gate_mean']=gate.detach().float().mean();diag['node']=ndbg
        if memory_ablation!='node_only' and not temporal.is_empty:
            valid=temporal.history_support_valid.any(dim=-1) if temporal.history_support_valid is not None and temporal.history_support_valid.shape[0]==temporal.tokens.shape[0] else torch.ones(len(temporal.tokens),device=temporal.tokens.device,dtype=torch.bool)
            tmsg,tdbg=self.base.tracklet_attention(norm,query_ref_um,query_batch_index,temporal.tokens,temporal.ref_um,temporal.ref_um,temporal.ref_um.new_zeros((len(temporal.tokens),)),temporal.batch_index,valid,dref_um,return_debug=return_debug,full_attention=full_attention)
            gate=torch.sigmoid(self.base.tracklet_gate(torch.cat([norm,tmsg],dim=-1)));out=out+gate*tmsg;used=True
            if tdbg is not None: tdbg['gate_mean']=gate.detach().float().mean();diag['tracklet']=tdbg
        if used: out=out+torch.sigmoid(self.base.ffn_gate)*self.base.ffn(self.base.ffn_norm(out))
        if return_debug: diag['routing_mode']=self.mode
        return out,(diag if return_debug else None)

## 4. Patch decoder forward to pass source-group metadata

In [ ]:
def patched_layer_forward(self,q,spatial_tokens,spatial_pos_um,support,dref_um,spatial_mask_features,temporal=None,*,memory_ablation='full',return_debug=False,full_attention=False):
    x=q.embeddings;n=self.self_norm(x);a,_=self.self_attn(n,n,n,key_padding_mask=q.padding_mask,need_weights=False);x=x+a;self.last_temporal_debug=None
    if self.query_memory_enabled and temporal is not None:
        valid=~q.padding_mask;fb=torch.arange(x.shape[0],device=x.device,dtype=torch.long)[:,None].expand_as(valid)
        tm,self.last_temporal_debug=self.temporal_fusion(x[valid],(q.references_cellscale*dref_um[:,None,None])[valid],fb[valid],temporal,dref_um,memory_ablation=memory_ablation,return_debug=return_debug,full_attention=full_attention,query_source_ids=q.source_instance_ids[valid],query_types=q.query_types[valid])
        if self.last_temporal_debug is not None:
            slots=torch.arange(x.shape[1],device=x.device,dtype=torch.long)[None].expand_as(valid);self.last_temporal_debug['query_slot_index']=slots[valid].detach();self.last_temporal_debug['query_type']=q.query_types[valid].detach();self.last_temporal_debug['source_instance_id']=q.source_instance_ids[valid].detach()
        y=x.clone();y[valid]=tm;x=y
    c=self.cross_attn(self.cross_norm(x),spatial_tokens,spatial_pos_um,q.references_cellscale*dref_um[:,None,None],support,q.padding_mask,dref_um);x=x+c;x=x+self.ffn(self.ffn_norm(x));x=x.masked_fill(q.padding_mask[...,None],0)
    ref0=q.references_cellscale;delta=self._bounded_center_delta(self.center(x),q);refs=ref0+delta;emb=self.mask_embed(x);masks=dot_mask_logits(emb,spatial_mask_features).masked_fill(q.padding_mask[...,None,None,None],-20.0)
    return replace(q,embeddings=x,references_cellscale=refs),{'exist_logits':self.exist(x).masked_fill(q.padding_mask,-20.0),'centers_cellscale':refs,'coarse_mask_logits':masks,'query_embeddings':x,'center_delta_cellscale':delta,'reference_before_update_cellscale':ref0}

def install_patch(model):
    model.query_builder.temporal_fusion=EventCompetitiveFusion(model.query_builder.temporal_fusion,model.cfg.temporal,EVENT_FEATURES,BASE_EVENT_PRIOR)
    for layer in model.query_decoder.layers:
        layer.temporal_fusion=EventCompetitiveFusion(layer.temporal_fusion,model.cfg.temporal,EVENT_FEATURES,BASE_EVENT_PRIOR);layer.forward=MethodType(patched_layer_forward,layer)
    model.to(device)
def fusions(model): return [model.query_builder.temporal_fusion,*[x.temporal_fusion for x in model.query_decoder.layers]]
def set_mode(model,mode):
    for x in fusions(model): x.set_mode(mode)

install_patch(model);set_mode(model,'baseline');model.eval()
print('Patch installed:',len(fusions(model)),'fusion sites')

## 5. Validate that `baseline` reproduces the unpatched model

In [ ]:
with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE): patched_baseline=model_forward_from_batch(model,b)
BASESIG=signature(patched_baseline)
def ma(a,c): return float((a.float()-c.float()).abs().max())
validation={'exist_max_abs':ma(UNPATCHED['exist'],BASESIG['exist']),'center_max_abs':ma(UNPATCHED['centers'],BASESIG['centers']),'query_embedding_max_abs':ma(UNPATCHED['qemb'],BASESIG['qemb']),'coarse_mean_abs':abs(UNPATCHED['coarse_mean']-BASESIG['coarse_mean']),'coarse_std_abs':abs(UNPATCHED['coarse_std']-BASESIG['coarse_std'])}
display(pd.DataFrame([validation]))
if validation['query_embedding_max_abs']>5e-3: raise RuntimeError('Baseline patch does not reproduce the original reader closely enough.')
del patched_baseline;gc.collect();torch.cuda.empty_cache();set_mode(model,'full')

## 6. Create the step-50 trainer

A fresh optimizer is used because the notebook adds new parameters. The model weights themselves start from the same Notebook-15 step-50 checkpoint.

In [ ]:
trainer=Trainer(model,cfg,device=device,amp_dtype='fp16');trainer.global_step=50
eval_criterion=RefinementCriterion(cfg.losses,cfg.queries,cfg.training).to(device).eval()
opt_ids={id(p) for g in trainer.optimizer.param_groups for p in g['params']}
new_params=[p for fu in fusions(model) for n,p in fu.named_parameters() if 'event.' in n or 'event_strength_raw' in n or 'log_temperature' in n]
assert all(id(p) in opt_ids for p in new_params)
print('step:',trainer.global_step,'new patch parameters:',sum(p.numel() for p in new_params))

## 7. Diagnostics

In [ ]:
def hard_dice(a,b,eps=1e-6):
    a=a.bool();b=b.bool();return float((2*(a&b).sum()+eps)/(a.sum()+b.sum()+eps))
def auc(scores,labels):
    s=np.asarray(scores,float);y=np.asarray(labels,int);p=y==1;n=y==0
    if not p.any() or not n.any(): return float('nan')
    r=rankdata(s);return float((r[p].sum()-p.sum()*(p.sum()+1)/2)/(p.sum()*n.sum()))
def offdiag(m):
    if m.shape[0]<2:return m.new_zeros(0)
    return m[~torch.eye(m.shape[0],dtype=torch.bool,device=m.device)]
def source_rows(out,layer=-1):
    d=out.debug['query_temporal_attention'][layer];qt=d['query_type'].detach().cpu();src=d['source_instance_id'].detach().cpu();rows=torch.nonzero((src==SOURCE_ID)&((qt==QUERY_PRIMARY)|(qt==QUERY_SPLIT)),as_tuple=False).flatten();return d,rows

def attention_metrics(out):
    d,rows=source_rows(out);w=d['node']['full_weights'][rows].detach().float().cpu();w=w/w.sum(-1,keepdim=True).clamp_min(1e-12);cos=F.normalize(w,dim=-1)@F.normalize(w,dim=-1).T;cv=offdiag(cos);top=w.argmax(-1);tracks=out.debug['node_memory']['tracklet_id'].detach().cpu().long();ent=-(w*w.clamp_min(1e-12).log()).sum(-1)/math.log(w.shape[-1]);em=w[:,CORRECTION_EVENT].sum(-1);cm=w[:,CONTINUOUS].sum(-1);bm=w[:,BOUNDARY].sum(-1)
    ef=float(CORRECTION_EVENT.float().mean());cf=float(CONTINUOUS.float().mean())
    return {'attention_cos_mean':float(cv.mean()),'attention_cos_median':float(cv.median()),'unique_top_nodes':int(torch.unique(top).numel()),'unique_top_tracklets':int(torch.unique(tracks[top]).numel()),'normalized_entropy_mean':float(ent.mean()),'correction_event_mass':float(em.mean()),'continuous_mass':float(cm.mean()),'boundary_mass':float(bm.mean()),'event_enrichment':float(em.mean()/max(ef,1e-8)),'continuous_enrichment':float(cm.mean()/max(cf,1e-8)),'mean_max_weight':float(w.max(-1).values.mean()),'competition_group_size':float(d['node']['competition_group_size'][rows].float().mean())}
def center_metrics(out):
    qt=out.query_types[0].detach().cpu();src=out.source_instance_ids[0].detach().cpu();idx=torch.nonzero((src==SOURCE_ID)&((qt==QUERY_PRIMARY)|(qt==QUERY_SPLIT)),as_tuple=False).flatten();c=out.centers_cellscale[0,idx].detach().float().cpu()*float(out.dref_um[0].detach().cpu());p=torch.pdist(c);return {'center_pair_mean_um':float(p.mean()),'center_pair_median_um':float(p.median()),'center_pair_max_um':float(p.max())}

In [ ]:
@torch.no_grad()
def native_metrics(out,probe):
    qt=out.query_types[0].detach().cpu();src=out.source_instance_ids[0].detach().cpu();idx_cpu=torch.nonzero((src==SOURCE_ID)&((qt==QUERY_PRIMARY)|(qt==QUERY_SPLIT)),as_tuple=False).flatten();idx=idx_cpu.to(device)
    with torch.autocast(device_type='cuda',dtype=AMP_DTYPE): logits=model.render_masks(out,[idx])[0].flatten(1)
    p=logits.float().sigmoid();s=p.sum(-1);pair=(2*(p@p.T)+1e-6)/(s[:,None]+s[None,:]+1e-6);vals=offdiag(pair);gt_map=target['label_map'].to(device=device,dtype=torch.int32).flatten();gt_ids=target['ids'].detach().cpu();assigned=[]
    for li,q in enumerate(idx_cpu.tolist()):
        ti=probe.query_to_target.get((0,int(q)))
        if ti is None: continue
        gt=(gt_map==int(gt_ids[ti])).float();dice=(2*(p[li]*gt).sum()+1e-6)/(p[li].sum()+gt.sum()+1e-6);assigned.append(float(dice))
    outm={'native_pair_dice_mean':float(vals.mean()),'native_pair_dice_median':float(vals.median()),'assigned_native_dice_mean':float(np.mean(assigned)) if assigned else float('nan')};del logits,p,gt_map;torch.cuda.empty_cache();return outm

def routing_params():
    rows=[]
    for name,fu in zip(['component','decoder0','decoder1','decoder2'],fusions(model)):
        logit,prob,res=fu.event();rows.append({'fusion':name,'event_strength':float(fu.event_strength.mean().detach().cpu()),'temperature':float(fu.temperature.detach().cpu()),'prob_event':float(prob[CORRECTION_EVENT.to(prob.device)].mean().detach().cpu()),'prob_continuous':float(prob[CONTINUOUS.to(prob.device)].mean().detach().cpu()),'residual_abs_mean':float(res.abs().mean().detach().cpu())})
    return pd.DataFrame(rows)

In [ ]:
@torch.no_grad()
def snapshot(tag,mode='full',compute_native=True):
    set_mode(model,mode);model.eval();gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();t=time.perf_counter()
    with torch.autocast(device_type='cuda',dtype=AMP_DTYPE): out=model_forward_from_batch(model,b,return_debug=True,return_full_temporal_attention=True);losses=eval_criterion(out,b['targets'])
    probe=run_matching_probe(out,b['targets']);match=probe.matches[0];valid=(~out.query_padding_mask[0]).detach().cpu();prob=out.exist_logits[0].sigmoid().detach().float().cpu();pos=torch.zeros_like(valid);pos[match.pred_indices.detach().cpu()]=True
    fg=out.dense_outputs['foreground_logits'][0,0].sigmoid().detach();bd=out.dense_outputs['boundary_logits'][0,0].sigmoid().detach();summary={'tag':tag,'step':int(trainer.global_step),'routing_mode':mode,'loss':float(losses['loss'].detach().cpu()),'foreground_dice':hard_dice(fg>0.5,target['foreground'].to(device)>0.5),'boundary_dice':hard_dice(bd>0.5,target['boundary'].to(device)>0.5),'coarse_dice_loss':float(losses['dice_coarse'].detach().cpu()),'native_dice_loss':float(losses['dice_hi'].detach().cpu()),'exist_auc':auc(prob[valid].numpy(),pos[valid].long().numpy()),'matched_gt':int(match.target_indices.numel()),'all_gt_matched':bool(match.target_indices.numel()==len(target['ids'])),'peak_cuda_gib':float(torch.cuda.max_memory_allocated()/1024**3),**attention_metrics(out),**center_metrics(out)}
    if compute_native: summary.update(native_metrics(out,probe))
    d,rows=source_rows(out);w=d['node']['full_weights'][rows].detach().float().cpu();top=w.argmax(-1);nm=out.debug['node_memory'];top_table=pd.DataFrame({'query_row':np.arange(len(rows)),'top_node':top.numpy(),'time':nm['time_offset'].detach().cpu()[top].round().long().numpy(),'tracklet':nm['tracklet_id'].detach().cpu()[top].numpy(),'weight':w.max(-1).values.numpy(),'correction_event':CORRECTION_EVENT[top].numpy(),'continuous':CONTINUOUS[top].numpy()});summary['elapsed_s']=time.perf_counter()-t
    return {'summary':summary,'top_table':top_table,'params':routing_params()}

## 8. Step-50 baseline vs the untrained patch

This shows the immediate effect of event prioritization + competition before any new learning.

In [ ]:
S={}
S['step50_baseline']=snapshot('step50_baseline','baseline')
S['step50_full_untrained']=snapshot('step50_full_untrained','full')
display(pd.DataFrame([S['step50_baseline']['summary'],S['step50_full_untrained']['summary']]))
print('Full-patch top nodes:');display(S['step50_full_untrained']['top_table']);display(S['step50_full_untrained']['params']);set_mode(model,'full')

## 9. Train only query/native/joint stages (50→85)

In [ ]:
BOUND={70:'step70_query',80:'step80_native',85:'step85_joint'};training=[]
def save_local(tag): torch.save({'model':model.state_dict(),'step':trainer.global_step,'tag':tag,'note':'Notebook17 runtime-patched model; reinstall patch before loading.'},RUN_DIR/f'{tag}.pt')
start=time.perf_counter();set_mode(model,'full')
while trainer.global_step<85:
    m=trainer.train_step(b);step=trainer.global_step;training.append({'step':step,'stage':trainer.curriculum_stage.name,**m})
    if step%LOG_EVERY==0 or step in BOUND or step==51: print(f"step {step:3d} | {trainer.curriculum_stage.name:17s} | loss={m['loss']:.5f} coarse={m.get('dice_coarse',float('nan')):.5f} native={m.get('dice_hi',float('nan')):.5f}")
    if step in BOUND:
        tag=BOUND[step];S[tag]=snapshot(tag,'full');save_local(tag);print('\n',tag);display(pd.DataFrame([S[tag]['summary']]));display(S[tag]['top_table']);display(S[tag]['params']);set_mode(model,'full');model.train()
print('elapsed min:',round((time.perf_counter()-start)/60,2))

## 10. Stage progression and Notebook-15 comparison

In [ ]:
stage_df=pd.DataFrame([x['summary'] for x in S.values()]);stage_df.to_csv(RUN_DIR/'stage_metrics.csv',index=False)
cols=['tag','loss','foreground_dice','boundary_dice','native_dice_loss','correction_event_mass','continuous_mass','event_enrichment','attention_cos_mean','unique_top_nodes','unique_top_tracklets','center_pair_median_um','native_pair_dice_mean','assigned_native_dice_mean']
display(stage_df[[c for c in cols if c in stage_df.columns]])
nb15=NB15_RUN/'stage_metrics.csv'
if nb15.exists(): print('Notebook 15 final rows:');display(pd.read_csv(nb15).tail(2))

## 11. Same-weight causal routing ablations at step 85

This is the decisive comparison because every row uses the same trained parameters.

In [ ]:
A={};rows=[]
for mode in ['full','no_event','no_competition','baseline','shuffled_event']:
    print('\n---',mode,'---');A[mode]=snapshot('ablation_'+mode,mode);rows.append(A[mode]['summary']);display(pd.DataFrame([A[mode]['summary']]))
ab=pd.DataFrame(rows);full_loss=float(ab.loc[ab.routing_mode=='full','loss'].iloc[0]);ab['loss_delta_vs_full']=ab.loss-full_loss;ab.to_csv(RUN_DIR/'same_weight_routing_ablations.csv',index=False)
display(ab[['routing_mode','loss','loss_delta_vs_full','native_dice_loss','correction_event_mass','continuous_mass','event_enrichment','attention_cos_mean','unique_top_nodes','unique_top_tracklets','center_pair_median_um','native_pair_dice_mean','assigned_native_dice_mean']]);set_mode(model,'full')

## 12. Final event relevance and top source-9 explanations

In [ ]:
params=routing_params();params.to_csv(RUN_DIR/'final_routing_parameters.csv',index=False);display(params)
tops=[]
for mode,res in A.items():
    t=res['top_table'].copy();t.insert(0,'routing_mode',mode);tops.append(t)
top_df=pd.concat(tops,ignore_index=True);top_df.to_csv(RUN_DIR/'source9_top_nodes_by_routing_mode.csv',index=False);display(top_df)

## 13. Automatic compact outcome report

A convincing result should satisfy most of these:

- `full` gives higher source-9 assigned Dice than `baseline`;
- `full` gives higher assigned Dice than `no_event` and `no_competition`;
- `full` gives higher assigned Dice than `shuffled_event`;
- correction-event attention is enriched above its population fraction;
- continuous-track mass is reduced;
- sibling attention cosine falls and unique top nodes increase;
- median center separation increases.

If attention diversifies but assigned Dice does not improve, the patch is **not** yet a success.

In [ ]:
def r(mode): return ab[ab.routing_mode==mode].iloc[0]
f=r('full');base=r('baseline');ne=r('no_event');nc=r('no_competition');sh=r('shuffled_event')
report={
 'full_minus_baseline_assigned_dice':float(f.assigned_native_dice_mean-base.assigned_native_dice_mean),
 'full_minus_no_event_assigned_dice':float(f.assigned_native_dice_mean-ne.assigned_native_dice_mean),
 'full_minus_no_competition_assigned_dice':float(f.assigned_native_dice_mean-nc.assigned_native_dice_mean),
 'full_minus_shuffled_event_assigned_dice':float(f.assigned_native_dice_mean-sh.assigned_native_dice_mean),
 'full_minus_baseline_attention_cos':float(f.attention_cos_mean-base.attention_cos_mean),
 'full_minus_baseline_event_mass':float(f.correction_event_mass-base.correction_event_mass),
 'full_event_enrichment':float(f.event_enrichment),
 'full_unique_top_nodes':int(f.unique_top_nodes),
 'full_center_median_um':float(f.center_pair_median_um),
}
print(json.dumps(report,indent=2));json.dump(report,open(RUN_DIR/'outcome_summary.json','w'),indent=2)
pd.DataFrame(training).to_csv(RUN_DIR/'training_log.csv',index=False)

## 14. Decision guide

### Commit-worthy signal

`full` must improve **biological specialization**, not merely make attention different:

```text
more event enrichment
+ more unique temporal explanations
+ larger sibling center separation
+ higher assigned source-9 Dice
```

### If event helps but competition does not

If `no_competition ≈ full > no_event`, keep event relevance and redesign the competition rule.

### If competition helps but event does not

If `no_event ≈ full > no_competition`, keep source-group competition but remove or weaken the event prior.

### If full still behaves like baseline

Then the diffuse Detection-GNN representation is probably the next bottleneck. The next experiment should change **GNN normalization/routing**, not discard historical nodes.

In [ ]:
print('Artifacts:')
for p in sorted(RUN_DIR.glob('*')): print(' ',p.name)
set_mode(model,'full');gc.collect();torch.cuda.empty_cache();print('CUDA cache cleared.')